In [3]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Abhi0072/HateXplain")

Repo card metadata block was not found. Setting CardData to empty.


In [4]:
# Access the 'train' split
train_dataset = ds['train']
# Convert to pandas DataFrame
df = train_dataset.to_pandas()
df.sample(10)

,text,label
19838,the only way you lose the white card is if you...,offensive
13531,drinking by myself how ghetto,offensive
14107,they need to be relentlessly shamed and litera...,normal
294,i probably should have put nsfw on that also n...,hatespeech
10830,<user> we have no self respect shameless left ...,offensive
14852,no democrats do not want them to assimilate as...,offensive
17604,<user> has got to be one of the dumbest muslim...,hatespeech
12992,<user> like i have been saying for awhile now ...,offensive
10009,oklahoma mudshark is attacked while holding he...,offensive
11796,you don ’ t never fuck yo niggas bitch n you d...,offensive


In [5]:
df.shape

(20148, 2)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20148 entries, 0 to 20147
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    20148 non-null  object
 1   label   20148 non-null  object
dtypes: object(2)
memory usage: 314.9+ KB


In [7]:
df['label'].value_counts()

,count
label,
normal,8209
hatespeech,5991
offensive,5948


In [8]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<user>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'>.*\n?', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["text"] = df["text"].apply(clean_text)

In [9]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2,stratify=y)

In [10]:
from sklearn.preprocessing import LabelEncoder
ln =  LabelEncoder()

y_train = ln.fit_transform(y_train)
y_test  = ln.transform(y_test)

In [11]:
print(ln.classes_)

['hatespeech' 'normal' 'offensive']


In [12]:
df["label"].value_counts()

,count
label,
normal,8209
hatespeech,5991
offensive,5948


In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 15000
max_len = 80

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post")

In [14]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))


In [15]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [16]:
from keras import Sequential
from keras.layers import Dense,Flatten,Dropout,Embedding,Bidirectional,Input,LSTM

In [17]:
embedding_dim = 200
embedding_index = {}
with open("glove.6B.200d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = vector

In [18]:
import numpy as np
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i >= vocab_size:
        continue
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [19]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM,
    Dense, Dropout, Attention, GlobalAveragePooling1D
)

inputs = Input(shape=(max_len,))

x = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    trainable=False
)(inputs)

x = Bidirectional(LSTM(64, return_sequences=True))(x)
x = Bidirectional(LSTM(32, return_sequences=True))(x)

# Attention
x = Attention()([x, x])

# Collapse sequence dimension
x = GlobalAveragePooling1D()(x)

x = Dense(32, activation='relu')(x)
x = Dropout(0.3)(x)

outputs = Dense(3, activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 80)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 80, 200)   │  3,000,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 80, 128)   │    135,680 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 80, 64)    │     41,216 │ bidirectional[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 80, 64)    │          0 │ bidirectional_1[… │
│ (Attention)         │                   │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ attention[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 3)         │         99 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,179,075 (12.13 MB)

 Trainable params: 179,075 (699.51 KB)

 Non-trainable params: 3,000,000 (11.44 MB)

In [20]:
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_test_pad, y_test),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/15
252/252 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.3872 - loss: 1.0847 - val_accuracy: 0.4501 - val_loss: 1.0643
Epoch 2/15
252/252 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.4793 - loss: 1.0366 - val_accuracy: 0.4878 - val_loss: 1.0262
Epoch 3/15
252/252 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.5106 - loss: 1.0037 - val_accuracy: 0.4916 - val_loss: 1.0140
Epoch 4/15
252/252 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.5281 - loss: 0.9801 - val_accuracy: 0.5189 - val_loss: 0.9965
Epoch 5/15
252/252 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.5477 - loss: 0.9518 - val_accuracy: 0.5174 - val_loss: 1.0073
Epoch 6/15
252/252 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.5704 - loss: 0.9244 - val_accuracy: 0.5226 - val_loss: 1.0089
Epoch 7/15
252/252 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.5851 - loss: 0.8931 - val_accuracy: 0.5169 - val_loss: 1.0124


In [21]:
label_map = {
    0: "hatespeech",
    1: "normal",
    2: "offensive"
}
df["label"] = df["label"].map(label_map)

In [22]:
def predict(sentence):
  sequence = tokenizer.texts_to_sequences([sentences])
  padded = pad_sequences(sequence,maxlen=max_len,padding='post')
  pred = model.predict(padded)

  class_id = np.argmax(pred)
  confidence = float(np.max(pred))

  return {
      'sentence':sentence,
      'label':label_map[class_id],
      'confidence':confidence
  }


In [23]:
sentences = 'play one game of overwatch after a month away very first game instantly has two players calling people faggots and to kill yourself oh boy i gotta just not play this game at all anymore'
predict(sentences)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step


{'sentence': 'play one game of overwatch after a month away very first game instantly has two players calling people faggots and to kill yourself oh boy i gotta just not play this game at all anymore',
 'label': 'normal',
 'confidence': 0.5743041634559631}

In [24]:
sentences = 'this sounds like a kike national socialist'
predict(sentences)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


{'sentence': 'this sounds like a kike national socialist',
 'label': 'hatespeech',
 'confidence': 0.5696954131126404}

In [25]:
sentences = 'she wanna meet carti that bitch is a barbie'
predict(sentences)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


{'sentence': 'she wanna meet carti that bitch is a barbie',
 'label': 'hatespeech',
 'confidence': 0.38125306367874146}

# saving the model

In [26]:
model.save("hateXplain_lstm.keras")

In [28]:
import pickle

with open("tokenizerX.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("max_lenX.pkl", "wb") as f:
    pickle.dump(max_len, f)

with open("label_mapX.pkl","wb") as f:
    pickle.dump(label_map,f)
